# 连通性与字段名核对

连接配置从 `vol_strategy.py` 读，改那里的 `MODE` 就行，这里不重复配置。

- `MODE = "dma"` — 直连 Rotman 服务器，Mac/任意系统可用，**不需要 RIT Client**
- `MODE = "client"` — 连本机 Windows RIT Client 的 `localhost:9999`，需要桌面版 Client 已启动并登录

注意：浏览器版 RIT 和 Mac app 都**不会**在本机开 `localhost:9999`，它们走的是 DMA。

In [ ]:
import requests
import vol_strategy as vs

session = requests.Session()
session.headers.update(vs.AUTHORIZATION)
print("MODE:", vs.MODE)
print("endpoint:", vs.API_ENDPOINT)

In [ ]:
# 1. 连通性诊断。不要用 resp.json()，401 的返回体可能不是 JSON。
try:
    resp = session.get(f"{vs.API_ENDPOINT}/case", timeout=8)
except Exception as e:
    print("连不上：", type(e).__name__)
    print("-> MODE=client 时：Windows 桌面版 RIT Client 没启动/没登录")
    print("-> 浏览器版和 Mac app 不提供 localhost:9999，请改用 MODE=\"dma\"")
else:
    print("HTTP", resp.status_code, resp.reason)
    print(resp.text[:400] or "(空)")
    if resp.status_code == 401:
        print("\n401 -> 凭证不对，或 Client 未登录到案例")

In [ ]:
# 2. 第一次拉新闻，不带游标
resp = session.get(f"{vs.API_ENDPOINT}/news", params={"limit": 20})
print(resp.status_code)
news = resp.json()
news

In [ ]:
# 3. 核对字段名是否为 news_id / period / tick / ticker / headline / body
if news:
    print(sorted(news[0].keys()))
    print(news[0])

In [ ]:
# 4. 增量拉取：只要 news_id 比游标大的
last_news_id = max(n["news_id"] for n in news) if news else 0
print("last_news_id =", last_news_id)

resp2 = session.get(f"{vs.API_ENDPOINT}/news", params={"after": last_news_id, "limit": 20})
print(resp2.status_code)
resp2.json()

In [ ]:
# 5. 用真实公告文本验证波动率解析
for n in sorted(news, key=lambda x: x["news_id"]):
    print(f"tick={n['tick']:>3}  {str(vs.parse_vol_from_news(n)):26}  {n['body'][:60]}")

state = vs.new_vol_state()
vs.apply_news_to_state(news, state)
print("\nstate ->", state)

In [ ]:
# 6. 核对 /securities 的字段名（build_signal_table 依赖 ticker/last/bid/ask/position）
sec = session.get(f"{vs.API_ENDPOINT}/securities").json()
print(sorted(sec[0].keys()))
sec[:3]